In [1]:
import requests
import json
import pandas as pd
from pprint import pprint
from bs4 import BeautifulSoup

In [2]:
TARGET_URL = "https://results.zone"

In [3]:
show_data = lambda data: print(f"{type(data)}\n\n{data}")

In [4]:
def get_html(url, cookies=None, headers=None, params=None, timeout=10):
    """
    Получает HTML-страницу по URL.
    
    :param url: URL для запроса
    :param cookies: dict, словарь с cookies (по умолчанию None)
    :param headers: dict, словарь с заголовками (по умолчанию None)
    :param headers: dict, словарь с параметрами (по умолчанию None)
    :param timeout: int, время ожидания ответа в секундах (по умолчанию 10)
    :return: объект response, если запрос успешен
    """

    try:
        response = requests.get(url, cookies=cookies, headers=headers, params=params, timeout=timeout)
        return response
    except Exception as e:
        print(f"{e}")
        return None

# Вариант №1. BeautifulSoup.

In [6]:
def get_soup(html):
    return BeautifulSoup(html.text, "html.parser")

In [ ]:
# Получаем html и готовим суп
html = get_html(TARGET_URL)
soup = get_soup(html)
show_data(soup)

### Разминка

In [ ]:
# DevTools: id="navbars", class_="collapse navbar-collapse"
navbars = soup.find("div", id="navbars")
show_data(navbars)

In [ ]:
links = navbars.find_all("a")
show_data(links)

In [ ]:
links = {i.get_text(): i.get("href") for i in links}
links

### Собираем данные. Попытка №1 

In [ ]:
# DevTools: "div", id="events_list"
table = soup.find("div", id="events_list")
show_data(table)

In [ ]:
# DevTools: container-fluid mb-4
for i in soup.body:
    print(f"{i}\n")

### Собираем данные. Попытка №2

In [15]:
# Штош, а если добавить больше человечности в запрос к сайту?
# В этом нам поможет https://curlconverter.com/

cookies = {
    '_ym_uid': '1733496338990587750',
    '_ym_d': '1733496338',
    '_ga': 'GA1.2.782272388.1733496338',
    '_gid': 'GA1.2.2074866395.1733496338',
    '_rz_session': 'WHZVQXJNOVRrZzlRK1pjRVpqUXhocWZQL1NRS3FBeStzMCtnVTdVdmJteHZuMFZrODNJSU9qWHAyYTZtOGROV1E3ZXA2cFBmZHJ4dmlOYThQdnBlaUZ3d0RhZEFqYXM0RzhxZ2R6R0tJTTRnZFZLNW1LYWhUT0dOWU0yWWpmNDdtbktzMXJvaWtRWjU5UWY3OW5HTW1RPT0tLTBLVzA4WGk2V1hUN1h0cnBTaCtYWFE9PQ%3D%3D--6afc2400b18aa4e6ecb5b701ad428779bcf8c93e',
    '_ga_DK3GTFFR9Q': 'GS1.2.1733568903.3.0.1733568903.60.0.0',
    '_ym_isad': '2',
}

headers = {
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'accept-language': 'ru,en;q=0.9',
    'cache-control': 'max-age=0',
    'cookie': '_ym_uid=1733496338990587750; _ym_d=1733496338; _ga=GA1.2.782272388.1733496338; _gid=GA1.2.2074866395.1733496338; _rz_session=WHZVQXJNOVRrZzlRK1pjRVpqUXhocWZQL1NRS3FBeStzMCtnVTdVdmJteHZuMFZrODNJSU9qWHAyYTZtOGROV1E3ZXA2cFBmZHJ4dmlOYThQdnBlaUZ3d0RhZEFqYXM0RzhxZ2R6R0tJTTRnZFZLNW1LYWhUT0dOWU0yWWpmNDdtbktzMXJvaWtRWjU5UWY3OW5HTW1RPT0tLTBLVzA4WGk2V1hUN1h0cnBTaCtYWFE9PQ%3D%3D--6afc2400b18aa4e6ecb5b701ad428779bcf8c93e; _ga_DK3GTFFR9Q=GS1.2.1733568903.3.0.1733568903.60.0.0; _ym_isad=2',
    'if-none-match': 'W/"140e6884a65e05ec5c3e3aec01890014"',
    'priority': 'u=0, i',
    'referer': 'https://results.zone/',
    'sec-ch-ua': '"Not/A)Brand";v="8", "Chromium";v="126", "YaBrowser";v="24.7", "Yowser";v="2.5"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"macOS"',
    'sec-fetch-dest': 'document',
    'sec-fetch-mode': 'navigate',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-user': '?1',
    'upgrade-insecure-requests': '1',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 YaBrowser/24.7.0.0 Safari/537.36',
}

In [ ]:
# Мы пытались, но не помогло
html = get_html(TARGET_URL, cookies=cookies, headers=headers)
soup = get_soup(html)
for i in soup.body:
    print(f"{i}\n")

### Собираем данные. Попытка №3

In [ ]:
# DevTools: не сдаемся и начинаем искать названия. Например, "Swimcup Минуты Декабрь" 
scripts = soup.head.find_all("script")
show_data(scripts)

In [ ]:
target_script = soup.head.find("script", string=lambda s: s and "general.events=[" in s)
show_data(target_script)

In [ ]:
events_string = target_script.string.split("general.events=")[1].split(";")[0]
show_data(events_string)

In [ ]:
events = json.loads(events_string)
df_events = pd.DataFrame(events)
df_events

### Выводы
__Плюсы__:
1. Быстро реализовать для разового проекта;

__Минусы__:
1. Все меньше сайтов, которые отдают данные в html;
2. Полностью зависит от верстки сайта;
3. Сложно поддерживать при масштабировании;
4. Необходимо отдельно собирать логику обхода страниц, если их больше чем одна;
5. Фильтровать, проверять, фильтровать, проверять, фильтровать, проверять, ...; 
6. ???

# Вариант №2. А можно как-то попроще, а?

In [ ]:
# DevTools: Fetch/XHR – типы запросов, которые браузер отправляет на сервер для получения данных
# evTools -> Fetch/XHR -> Preview
url = f"{TARGET_URL}/?q[sport_eq]=1"
response = get_html(url=url)
response.url

In [ ]:
# DevTools: Fetch/XHR + Content-Type
url = f"{TARGET_URL}/?q[sport_eq]=1"
params = {"format": "json"}
response = get_html(url=url, params=params)
show_data(response.json())

In [ ]:
events = response.json()
df_events = pd.DataFrame.from_dict(events)
df_events

In [ ]:
# Вспомним нашу цель – результаты соревнований!
# https://results.zone/swimmasters-rostov-2024
target = events[-1::][0]
pprint(target)

In [ ]:
# Соберем ссылки на результаты и прокачаем их добавив .json
links = [f"{TARGET_URL}{i['self']}.json" for i in target["races"]]
links

In [ ]:
# Рассмотрим первую ссылку
html = get_html(links[0]).json()
html.keys()

In [ ]:
html["items"][0]

In [ ]:
html["page_info"]

In [29]:
# Собираем результаты одного соревнования
df_event_results = pd.DataFrame()
for i in links:
    html = get_html(i).json()
    df_tmp = pd.DataFrame.from_dict(html["items"])
    df_event_results = pd.concat([df_event_results, df_tmp], axis=0)

In [ ]:
df_event_results 

In [ ]:
# Бонус. Мы всегда можем понять все ли мы собрали. Рассмотрим другой пример
url = "https://results.zone/swimcup-gora-2024/races/7431/results.json"
example_html = get_html(url)
example = example_html.json()
example['page_info']

### Выводы
__Плюсы__:
1. Масштабируемость;
2. Не зависит от верстки сайта;
3. Не сложно реализовать логику обхода "страниц" в БД;
4. Не сложно автоматизировать и поддерживать;
5. ??? 

__Минусы__:
1. Необходимо потратить время на изучение сайта и его запросов к БД;
2. Если жадничать (много потоков и без timeout), то можно словить временный бан;
3. REST API, GraphQL, ...?

__Доп.материалы:__
1. Олег Молчанов – [YouTube. "Асинхронность в Python"](https://www.youtube.com/playlist?list=PLlWXhlUMyooawilqK4lPXRvxtbYiw34S8)
2. Олег Молчанов – [Boosty. "Асинхронный Python и Asyncio"](https://boosty.to/omolchanov/posts/34ef77a6-e947-4505-8d81-9667276448ba)
3. Фаулер Мэттью – [Книга. "Asyncio и конкурентное программирование на Python"](https://dmkpress.com/catalog/computer/programming/python/978-5-93700-166-5/)  

# Подход №3. Playwright.

In [90]:
# playwright codegen https://results.zone/
# playwright codegen https://www.youtube.com/ 

### Выводы
__Плюсы__:
1. Действия с элементами на странице: клики по таймеру/условию, скроллы, заполнение форм и тд;
2. Автоматическая проверка изменения состояния на странице (Assertions);
3. Перехват и изменение сетевых запросов;
4. Масштабируемость;
5. Меньше кода чем в Selenium;
6. ??? 

__Минусы__:
1. Зависит от верстки сайта;
2. Не так сложно поддерживать, как BS при масштабировании, но из-за предыдущего пункта могут быть сложности;

__Доп.материалы:__
1. Stepik – ["Автоматизация тестирования с помощью Playwright Python"](https://stepik.org/course/128626/syllabus)
2. [playwright.dev](https://playwright.dev/python/docs/intro)


# Подход №4. API.
- __YouTube__:
    - [API](https://developers.google.com/youtube/v3/docs)
    - [Quota](https://developers.google.com/youtube/v3/determine_quota_cost)
- __VK__:
    - [API](https://dev.vk.com/ru/api/overview)
    - [API: execute & VKScript-код](https://dev.vk.com/ru/method/execute)    
    ```
    code = f"""
        var i = 0;
        var users = [{users}];
        var walls = [];
        var results = [];
        
        while (i < users.length) {{
            walls = API.wall.get({{'owner_id': users[i], 'count': 10, 'offset': 0, 'filter': 'owner'}});
            results.push(walls.items);                    
            i = i + 1;
        }};
        
        return results;"""
    ```
